# 03 · Piper Tatar: экспериментальное дообучение на CPU

**Экспериментальный блокнот.** Использует `IS2AI/TatarTTS` и **старую** систему `rhasspy/piper`, которая архивирована в 2025 году и несовместима с некоторыми новыми зависимостями Colab. Перед нами рецепт с проверками, а не гарантированно отработанное обучение в Colab 2026. Рекомендуется сначала проверить F5-TTS без обучения.

Нужны: `MyDrive/tatar_audio_lab/corpus_22050/consented_speaker_01/metadata.csv`, подпапка `wav`, мужской **training .ckpt** из официального [архива TatarTTS](https://drive.google.com/drive/folders/1YmtDVYLVogEfw3SE7GUl0LHZaIojAeap?usp=sharing). Не путать с `.onnx`. Права на голос и исходные записи обязательны.

**План для Colab:** CPU preflight → отдельная среда Python 3.10 → установка legacy Piper → препроцессинг → `--resume_from_checkpoint` → периодическое сохранение в Google Drive → экспорт ONNX. При нехватке памяти уменьшить batch-size до 4 или 2.

In [ ]:
from pathlib import Path
import os, sys, subprocess, torch
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
torch.set_num_threads(4)
print('Device: CPU; precision: float32; threads:', torch.get_num_threads())
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/tatar_audio_lab')
CORPUS=ROOT/'corpus_22050'/'consented_speaker_01'
CKPT_DIR=ROOT/'checkpoints'; CKPT_DIR.mkdir(parents=True,exist_ok=True)
assert (CORPUS/'metadata.csv').exists(), 'Сначала соберите разрешённый и вычитанный корпус в Notebook 01.'
assert (CORPUS/'wav').is_dir(), 'Отсутствует wav/'
print('Датасет:',CORPUS)
print('CKPT files:',[p.name for p in CKPT_DIR.glob('*.ckpt')])


In [ ]:
# Защитные флаги и проверка корпуса. Нельзя обучать модель актёра по публикациям без необходимых разрешений.
RIGHTS_TO_TRAIN_AND_SYNTHESIZE = False
assert RIGHTS_TO_TRAIN_AND_SYNTHESIZE, 'Нужно подтверждение разрешений на использование голоса и исполнений.'
META=(CORPUS/'metadata.csv').read_text(encoding='utf-8').splitlines()
assert len(META)>=30, f'Слишком мало примеров: {len(META)}. Рекомендуется 100+ вычитанных сегментов.'
for n,line in enumerate(META):
    parts=line.split('|')
    assert len(parts)==2 and parts[1].strip(), f'Ошибка metadata.csv в строке {n+1}'
    assert (CORPUS/'wav'/(parts[0]+'.wav')).exists(), f'Нет WAV: {parts[0]}'
CKPTS=sorted(CKPT_DIR.glob('*.ckpt'))
assert CKPTS, 'Загрузите совместимую контрольную точку мужского татарского Piper в MyDrive/tatar_audio_lab/checkpoints/'
PRETRAINED=CKPTS[0]
print('Примеров:',len(META),'Веса:',PRETRAINED)


### Подготовка отдельной среды

Старое `rhasspy/piper` исторически собирается под Python 3.10. Этот шаг использует `uv` для получения отдельного Python без замены системного Colab. Зависимости legacy могут потребовать исправлений; установка и обучение **не были испытаны на GPU в этой беседе**.

In [ ]:
# Следующие ячейки выполняются только в Colab с доступом в интернет.
import subprocess, shutil
subprocess.run(['apt-get','update','-qq'],check=True,stdout=subprocess.DEVNULL)
subprocess.run(['apt-get','install','-y','-qq','espeak-ng','python3-dev','build-essential','cmake'],
               check=True,stdout=subprocess.DEVNULL)
subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
subprocess.run(['uv','python','install','3.10'],check=True)
subprocess.run(['uv','venv','/content/piper310','--python','3.10'],check=True)
PY='/content/piper310/bin/python'
print('Legacy python:',subprocess.check_output([PY,'--version'],text=True).strip())


In [ ]:
PIPER=Path('/content/piper')
if not PIPER.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/rhasspy/piper.git',str(PIPER)],check=True)
# pip<=24 для устаревших metadata в lightning. Установка may require manual fixes on evolving Colab.
subprocess.run(['uv','pip','install','--python',PY,'pip==24.0','wheel','setuptools','cython','numpy==1.24.4'],check=True)
subprocess.run([PY,'-m','pip','install','--no-build-isolation','pytorch-lightning==1.7.7',
                'torchmetrics==0.11.4','librosa==0.10.1','onnx','onnxruntime'],check=True)
subprocess.run([PY,'-m','pip','install','--no-build-isolation','-e',str(PIPER/'src/python')],check=True)
script=PIPER/'src/python/build_monotonic_align.sh'
assert script.exists(), 'Скрипт сборки не найден — интерфейс Piper изменился.'
subprocess.run(['bash',str(script)],cwd=PIPER/'src/python',check=True)
print('Зависимости Piper установлены. Проверьте GPU совместимость PyTorch перед запуском.')


In [ ]:
# Преобразуем проверенный корпус в формат старого Piper.
PROCESSED=Path('/content/piper_processed')
PROCESSED.mkdir(exist_ok=True)
cmd=[PY,'-m','piper_train.preprocess','--language','tt',
     '--input-dir',str(CORPUS),'--output-dir',str(PROCESSED),
     '--dataset-format','ljspeech','--single-speaker','--sample-rate','22050']
print('Запуск:', ' '.join(cmd))
subprocess.run(cmd,cwd=PIPER/'src/python',check=True)
assert (PROCESSED/'config.json').exists(), 'Piper preprocessing не создал config.json'
print('Preprocessing completed.')


### Fine-tuning с сохранением

**КРИТИЧНО:** возобновление обучения из `.ckpt` зависит от архитектуры, фонемного словаря и версии `pytorch-lightning`. Оригинальные авторы TatarTTS использовали конкретную ветку Piper. При `state_dict mismatch` нельзя просто продолжать с несовместимым checkpoint.

Контрольные точки нужно сохранять на Drive; прямое обучение в Drive может быть медленным, поэтому после короткого эксперимента переносите папку результатов. Максимальное число эпох здесь преднамеренно небольшое.

In [ ]:
RUN_TRAINING = False  # осознанно включите только после успешного препроцессинга
BATCH_SIZE=1
OUTPUT=ROOT/'piper_finetune_runs'
OUTPUT.mkdir(exist_ok=True)
if RUN_TRAINING:
    cmd=[PY,'-m','piper_train', '--dataset-dir',str(PROCESSED),
        '--accelerator','cpu','--devices','1','--batch-size',str(BATCH_SIZE),
        '--num-workers','0','--validation-split','0.05','--num-test-examples','0',
        '--max_epochs','40','--checkpoint-epochs','2','--precision','32',
        '--resume_from_checkpoint',str(PRETRAINED),'--default_root_dir',str(OUTPUT)]
    print(' '.join(cmd))
    subprocess.run(cmd,cwd=PIPER/'src/python',check=True)
else:
    print('RUN_TRAINING=False. Настройте CPU/GPU зависимости и права прежде, чем начинать обучение.')


In [ ]:
# Экспорт: путь до полученной НОВОЙ контрольной точки укажите вручную.
NEW_MODEL_CKPT=''  # например /content/drive/MyDrive/tatar_audio_lab/piper_finetune_runs/...ckpt
if NEW_MODEL_CKPT:
    assert Path(NEW_MODEL_CKPT).exists()
    dest=ROOT/'piper_finetune_runs'/'custom_tatar_voice.onnx'
    subprocess.run([PY,'-m','piper_train.export_onnx',NEW_MODEL_CKPT,str(dest)],
                   cwd=PIPER/'src/python',check=True)
    shutil.copy2(PROCESSED/'config.json',Path(str(dest)+'.json'))
    print('Exported:',dest)
else:
    print('Экспорт не выполнялся: ещё нет обученного .ckpt')


### Альтернативы

Если `rhasspy/piper` ломается на зависимостях в современной Colab, это не недостаток GPU. Можно запускать его в Docker/среде с зафиксированными версиями CUDA/Python или переходить на поддерживаемый тренировочный стек. Приоритетнее проверить современную F5-TTS без обучения в Notebook 02.

Источники: https://github.com/IS2AI/TatarTTS, https://github.com/rhasspy/piper/blob/master/TRAINING.md, https://research.google.com/colaboratory/faq.html